# Lab 3: Vector Search with Neo4jContextProvider

In this lab, you will build a self-grounding agent that uses vector search to find semantically similar movies in Neo4j and inject them as context before every LLM call. Vector search converts text into numerical embeddings that capture meaning, so the agent can find relevant content even when the user's words don't exactly match what's stored in the database.

## What you will learn

- How to create an embedder with `get_embedder()`
- How to configure `Neo4jContextProvider` with `index_type="vector"`
- How the provider automatically injects search results as context
- The difference between agent responses with and without context

## Setup

Load the environment variables from your `.env` file.

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

provider_name = os.getenv("LLM_PROVIDER", "openai").lower()
if provider_name == "azure":
    assert os.getenv("AZURE_OPENAI_API_KEY"), "AZURE_OPENAI_API_KEY not set in .env file"
    assert os.getenv("AZURE_OPENAI_ENDPOINT"), "AZURE_OPENAI_ENDPOINT not set in .env file"
else:
    assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not set in .env file"
assert os.getenv("NEO4J_URI"), "NEO4J_URI not set in .env file"
print("Environment loaded successfully")

## Import Dependencies

Import the required modules from the Microsoft Agent Framework and the Neo4j integration.

In [ ]:
from llm_provider import get_client, get_embedder
from agent_framework_neo4j import Neo4jContextProvider, Neo4jSettings

## Load Neo4j Settings

`Neo4jSettings` reads connection details and index names from environment variables (`NEO4J_URI`, `NEO4J_USERNAME`, `NEO4J_PASSWORD`, `NEO4J_VECTOR_INDEX_NAME`).

In [ ]:
neo4j_settings = Neo4jSettings()

## Create an Embedder

Vector search needs an embedder to convert text into vector representations. Embeddings capture semantic meaning, so a query about "machines gaining consciousness" can match movies about AI even without those exact words. The `get_embedder()` function returns the appropriate embedder for your configured provider, using `text-embedding-3-small` by default.

In [ ]:
embedder = get_embedder()

## Create the Context Provider

On each turn, the provider embeds the user's query, searches the Neo4j vector index for semantically similar movie plots, and injects the matches as context before the LLM responds. Configure `Neo4jContextProvider` with:
- Neo4j connection details from `neo4j_settings`
- `index_name` set to `neo4j_settings.vector_index_name`
- `index_type="vector"`
- The embedder you created above
- `top_k=5` to retrieve 5 results
- A `context_prompt` to guide the LLM on how to use the injected data

In [ ]:
provider = Neo4jContextProvider(
    uri=neo4j_settings.uri,
    username=neo4j_settings.username,
    password=neo4j_settings.get_password(),
    index_name=neo4j_settings.vector_index_name,
    index_type="vector",
    embedder=embedder,
    top_k=5,
    context_prompt=(
        "## Semantic Search Results\n"
        "Use the following semantically relevant movie plots from the "
        "knowledge graph to answer questions about movies:"
    ),
)

## Run the Agent

The provider connects to Neo4j using an async context manager (`async with provider`). Create an agent with the provider and run a query about movies.

In [ ]:
async with provider:
    client = get_client()

    agent = client.as_agent(
        name="vector-agent",
        instructions=(
            "You are a helpful assistant that answers questions about "
            "movies using the provided semantic search context. "
            "Be concise and accurate. When the context contains "
            "relevant information, cite it in your response."
        ),
        context_providers=[provider],
    )

    session = agent.create_session()

    query = "Find me movies about time travel"
    print(f"User: {query}\n")
    print("Answer: ", end="", flush=True)
    response = await agent.run(query, session=session)
    print(response.text)
    print()

## Summary

You created a `Neo4jContextProvider` with `index_type="vector"` that embeds the user's query, searches the `moviePlots` vector index in Neo4j, and injects matching results as context messages before the LLM generates a response. The agent receives relevant movie data automatically without calling any tool.

**What's next:** In the next lab, you will add graph-enriched retrieval to traverse relationships after the vector search, giving the agent structured metadata like genres, actors, and directors alongside plot text.